# 랭그래프(LangGraph) 기본 개념

랭그래프는 에이전트의 동작을 **그래프**로 설계하는 프레임워크다. 그래프는 세 가지로 이루어진다.

| 구성 요소 | 역할 |
|---|---|
| 상태(State) | 그래프 전체가 공유하는 데이터. 노드들이 읽고 갱신한다 |
| 노드(Node) | 실제 작업을 하는 함수. 상태를 받아서 바꿀 값을 반환한다 |
| 엣지(Edge) | 노드 사이의 실행 순서. 조건에 따라 다음 노드를 고르는 조건부 엣지도 있다 |

상태를 정의하는 법(`TypedDict` vs `Pydantic`), 상태 값을 합치는 리듀서(`add`, `add_messages`), 노드와 엣지를 추가하는 법을 순서대로 실습한다.

## 1. 환경변수 로드

In [1]:
from dotenv import load_dotenv

# .env의 API 키를 환경변수로 불러온다.
load_dotenv()

True

## 2. 그래프의 상태 정의하기

### 2-1. TypedDict로 상태 정의

랭그래프의 상태는 보통 `TypedDict`로 정의한다. 키 이름과 값의 타입을 적어두는 딕셔너리 설계도다. 아래 `State`는 문자열 리스트인 `messages` 하나를 가진다.

In [3]:
from typing import TypedDict

# 그래프 상태: messages라는 키에 문자열 리스트를 담는다.
class State(TypedDict):
    messages: list[str]

### 2-2. TypedDict는 타입을 검사하지 않는다

`TypedDict`의 타입 표시는 편집기·타입 검사기를 위한 힌트일 뿐, 실행할 때 검사하지 않는다. `name`에 문자열 대신 숫자 `123`을 넣어도 에러 없이 그대로 만들어진다.

In [4]:
from typing import TypedDict 

# 사용자 정보를 담는 딕셔너리의 구조(키와 타입)를 정의
class User(TypedDict):
    id: int 
    name: str 
    email: str 

user1: User = {
    'id': 1,
    'name': 'nayeon_park',
    'email': 'example@gmail.com'
}
print(user1)

{'id': 1, 'name': 'nayeon_park', 'email': 'example@gmail.com'}


In [5]:
# name은 str로 정의했지만 int(123)를 넣어도 에러가 나지 않는다. (실행 시 타입 검사 없음)
user1: User = {
    'id': 1,
    'name': 123,
    'email': 'example@gmail.com'
}
print(user1)

{'id': 1, 'name': 123, 'email': 'example@gmail.com'}


### 2-3. Pydantic은 실행할 때 타입을 검사한다

같은 구조를 Pydantic `BaseModel`로 만들면, 객체를 만드는 순간 값의 타입을 검사한다. 올바른 값은 그대로 만들어지고, `name`에 숫자를 넣으면 `ValidationError`가 난다. 입력 데이터의 형식을 확실히 보장해야 할 때는 Pydantic이 안전하고, 가볍고 빠르게 쓰려면 `TypedDict`가 편하다.

In [6]:
from pydantic import BaseModel 

# 같은 구조를 Pydantic 모델로 정의 (객체 생성 시 타입 검사)
class User(BaseModel):
    id: int 
    name: str 
    email: str 

user_data = {
    'id': 1,
    'name': 'nayeon_park',
    'email': 'example@gmail.com'
}

# 딕셔너리를 키워드 인자로 풀어서(**) 객체 생성
user1 = User(**user_data)
print(user1)

id=1 name='nayeon_park' email='example@gmail.com'


In [7]:
# name에 int를 넣으면 Pydantic이 ValidationError를 낸다. (일부러 에러를 확인하는 셀)
user_data = {
    'id': 1,
    'name': 123,
    'email': 'example@gmail.com'
}

user = User(**user_data)
print(user)

ValidationError: 1 validation error for User
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

## 3. 상태에 리듀서(reducer) 함수 추가하기

노드가 상태의 어떤 키에 새 값을 반환하면, 기본적으로는 **기존 값을 덮어쓴다**. `Annotated[타입, 리듀서]`로 리듀서 함수를 지정하면 기존 값과 새 값을 **합치는 방식**을 정할 수 있다.

### 3-1. 직접 만든 add 리듀서

`add(left, right)`는 기존 값(`left`)과 새 값(`right`)을 `+`로 이어 붙인다. 리스트라면 새 메시지가 뒤에 계속 쌓인다.

In [8]:
from typing import TypedDict, Annotated 

# 리듀서: 기존 값(left)과 노드가 반환한 새 값(right)을 합치는 함수
def add(left, right):
    return left + right 

# messages가 갱신될 때 덮어쓰지 않고 add로 이어 붙인다.
class State(TypedDict):
    messages: Annotated[list[str], add]

### 3-2. 랭그래프 내장 리듀서 add_messages

대화 메시지를 다룰 때는 랭그래프의 `add_messages`를 쓴다. 단순히 이어 붙이는 `add`와 달리 **메시지 `id`를 기준으로 동작**한다.

- `id`가 다르면 → 뒤에 추가한다.
- `id`가 같으면 → 기존 메시지를 새 메시지로 **교체**한다.

In [9]:
from langchain_core.messages import AIMessage, HumanMessage 
from langgraph.graph.message import add_messages 

# id가 서로 다른 두 메시지 -> 리스트에 둘 다 남는다.
msgs1 = [HumanMessage(content="Hello", id="1")]
msgs2 = [AIMessage(content="Hi there!", id="2")]

add_messages(msgs1, msgs2)

[HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='1'),
 AIMessage(content='Hi there!', additional_kwargs={}, response_metadata={}, id='2', tool_calls=[], invalid_tool_calls=[])]

In [10]:
# id가 같은 메시지 -> 기존 "Hello"가 "Hello again"으로 교체된다.
msgs1 = [HumanMessage(content="Hello", id="1")]
msgs2 = [HumanMessage(content="Hello again", id="1")]

add_messages(msgs1, msgs2)

[HumanMessage(content='Hello again', additional_kwargs={}, response_metadata={}, id='1')]

챗봇·에이전트의 상태는 보통 이렇게 정의한다. `AnyMessage`는 `HumanMessage`, `AIMessage`, `SystemMessage`, `ToolMessage` 등 모든 메시지 타입을 받을 수 있다.

In [11]:
from langchain_core.messages import AnyMessage 
from langgraph.graph.message import add_messages 
from typing import TypedDict, Annotated 

# 에이전트에서 가장 많이 쓰는 상태 정의: 모든 종류의 메시지를 add_messages로 누적
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

## 4. 그래프의 실행 단위, 노드 추가하기

`StateGraph(State)`로 상태 구조를 가진 그래프를 만들고, `add_node(이름, 함수)`로 노드를 추가한다. 노드 함수는 현재 상태를 받아서 **바꿀 키만 담은 딕셔너리**를 반환한다. 여기서는 `messages`에 `operator.add` 리듀서를 걸어서, 노드가 반환한 답변이 기존 메시지 뒤에 추가된다.

In [21]:
from typing import TypedDict, Annotated 
from operator import add 

from langgraph.graph import StateGraph 

# operator.add도 리스트를 + 로 이어 붙이는 리듀서로 쓸 수 있다.
class State(TypedDict):
    messages: Annotated[list[str], add]

# State 구조를 가진 그래프 생성
graph = StateGraph(State)

# 노드 함수: 현재 상태를 받아서 갱신할 값을 반환
def chatbot(state: State):
    question = state["messages"]
    answer = f"사용자 입력을 그대로 반환하는 챗봇입니다. {question}라는 질문을 받았습니다."
    # messages에 answer를 추가 (리듀서 add로 기존 리스트 뒤에 붙음)
    return {"messages": [answer]}

# "chatbot"이라는 이름으로 노드 등록
graph.add_node("chatbot", chatbot)

## 5. 그래프의 실행 경로, 엣지 추가하기

`add_edge(출발, 도착)`로 노드 사이의 실행 순서를 연결한다. `START`는 그래프의 시작점, `END`는 끝점을 나타내는 특수 노드다. 아래는 `START → chatbot → END` 순서로 실행되는 가장 단순한 그래프다.

In [22]:
from langgraph.graph import START, END

# 시작 -> chatbot -> 끝
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

노드 이름 두 개를 넘기면 그 순서대로 이어진다. (`node_a`, `node_b`는 문법을 보여주기 위한 예시 이름이라 실제로 등록된 노드는 아니다.)

In [23]:
# 문법 예시: node_a 실행 후 node_b 실행 (실제로는 없는 노드라 compile하면 에러가 난다)
graph.add_edge("node_a", "node_b")

## 6. 그래프에 조건부 엣지 추가하기

`add_conditional_edges(출발 노드, 라우팅 함수, 매핑)`은 라우팅 함수의 반환값에 따라 다음 노드를 고른다. 아래는 chatbot의 마지막 메시지가 1000자를 넘으면 `summary` 노드로, 아니면 `END`로 가도록 분기한다. (`summary` 노드도 예시라 아직 만들지 않았다.)

In [24]:
# 라우팅 함수: 상태를 보고 어느 쪽으로 갈지 결정하는 값을 반환
def routing_function(state: State):
    # 마지막 메시지가 1000자를 넘으면 True
    if len(state["messages"][-1]) > 1000:
        return True 
    return False 

graph.add_conditional_edges(
    "chatbot",
    routing_function,
    # 반환값 -> 다음 노드 매핑 (True면 summary 노드로, False면 종료)
    {True: "summary", False: END}
)

## 정리

| 개념 | 코드 | 핵심 |
|---|---|---|
| 상태 정의 | `class State(TypedDict)` | 그래프가 공유하는 데이터 구조. `TypedDict`는 실행 시 타입 검사 없음, Pydantic `BaseModel`은 검사함 |
| 리듀서 | `Annotated[list[str], add]` | 새 값을 덮어쓰지 않고 기존 값과 합치는 방식 지정 |
| 메시지 리듀서 | `Annotated[list[AnyMessage], add_messages]` | `id`가 다르면 추가, 같으면 교체 |
| 그래프 생성 | `StateGraph(State)` | 상태 구조를 가진 그래프 |
| 노드 | `graph.add_node("이름", 함수)` | 상태를 받아 바꿀 값만 반환하는 함수 |
| 엣지 | `graph.add_edge(START, "chatbot")` | 노드 실행 순서 연결 (`START`, `END`는 시작·끝) |
| 조건부 엣지 | `graph.add_conditional_edges(노드, 라우팅 함수, 매핑)` | 상태에 따라 다음 노드를 분기 |

오늘은 그래프를 설계하는 문법까지만 실습했고, `graph.compile()`로 실제 실행 가능한 그래프를 만들어 `invoke()`하는 건 다음 실습(5.3 랭그래프로 에이전트 설계하고 구현하기)에서 이어진다.